In [ ]:
import pandas as pd
import warnings 
warnings.filterwarnings('ignore')

In [ ]:
data=pd.read_excel('??14??????????????.xlsx')
data.head()

In [ ]:
data.shape

In [ ]:
f=open('??15?????????.csv')
temp1=pd.read_csv(f,encoding='utf-8')
temp1.head()

In [ ]:
temp1.apply(lambda x:x['????Block']+'-'+str(x['??']),axis=1).unique()

In [ ]:
data.apply(lambda x:x['????(plot)']+'-'+str(x['year']),axis=1).unique()

In [ ]:
temp1['inner']=temp1.apply(lambda x:x['????Block']+'-'+str(x['??']),axis=1)
data['inner']=data.apply(lambda x:x['????(plot)']+'-'+str(x['year']),axis=1)

In [ ]:
data2=pd.merge(data,temp1,on='inner')

In [ ]:
data2

In [ ]:
data2.isnull().sum()

In [ ]:
data2=data2[[ '??', '????(plot)', '????(intensity)', 'SOC?????', 'SIC?????',
       'STC????', '??N', '??C/N?',  '??', '??',  '????',
       '???????',  '??', '???',  '?/??',
       '??(g)', '??(g)']]

In [ ]:
data2

In [ ]:
data2.shape

In [ ]:
temp2=pd.read_excel('??3?????2022?2012?.xls')
temp2.head()

In [ ]:
temp3=pd.read_excel('??4??????2012?2022?.xls')
temp3.head()

In [ ]:
temp4=pd.read_excel('??6?????-NDVI2012-2022?.xls')
temp4.head()

In [ ]:
temp5=pd.read_excel('??9????2012-2022?.xlsx')
temp5.head()

In [ ]:
tempdf=pd.merge(temp2,temp3)
tempdf.head()

In [ ]:
tempdf

In [ ]:
tempdf=pd.merge(tempdf,temp4)
tempdf=pd.merge(tempdf,temp5)

In [ ]:
tempdf

In [ ]:
data2=data2[['??', '????(plot)', '????(intensity)', 'SOC?????', 'SIC?????',
       'STC????', '??N', '??C/N?', '??', '??', '????', '???????', '??', '???',
       '?/??', '??(g)', '??(g)']]

In [ ]:
data2.head()

In [ ]:
pd.get_dummies(data2['????(plot)'],prefix='plot_')

In [ ]:
data2=pd.concat([data2,pd.get_dummies(data2['????(plot)'],prefix='plot_')],axis=1)

In [ ]:
data2['????(intensity)'].unique()

In [ ]:
#??(NG,?0?/?/???)???????(LGI,?2?/?/???)???????(MGI,4?/?/???)???????(HGI,8?/?/???)
data2['????']=data2['????(intensity)'].map(
{
    'NG':0,
    'LGI':1, 
    'MGI':2, 
    'HGI':3
})

In [ ]:
data2=pd.concat([data2,pd.get_dummies(data2['??'],prefix='??_')],axis=1)

data2=pd.concat([data2,pd.get_dummies(data2['??'],prefix='??_')],axis=1)

data2=pd.concat([data2,pd.get_dummies(data2['???????'],prefix='???_')],axis=1)

In [ ]:
data2.columns

In [ ]:
data2_process=data2[['??', '????', 'SOC?????', 'SIC?????',
       'STC????', '??N', '??C/N?',  '???',
       '?/??', '??(g)', '??(g)', 'plot__G11', 'plot__G12', 'plot__G13',
       'plot__G16', 'plot__G17', 'plot__G18', 'plot__G19', 'plot__G20',
       'plot__G21', 'plot__G6', 'plot__G8', 'plot__G9', '??__??', '??__?????',
       '??__?????', '??__?????', '??__?????', '??__??(6?)', '??__??(0?)',
       '??__??(3?)', '??__??(12?)', '???__AB', '???__PB', '???__PF',
       '???__PR']]

In [ ]:
df=pd.merge(tempdf,data2_process,on=['??'])

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()/df.shape[0]

In [ ]:
for i in ['???','?/??', '??(g)']:
    df[i]=df[i].fillna(df[i].mean())

In [ ]:
import numpy as np
def filter_extreme_MAD(dataframe,n=3):
    # MAD:????????
    for i in dataframe.columns:
        median = dataframe[i].quantile(0.5)
        new_median = ((dataframe[i] - median).abs()).quantile(0.50)
        max_range = median + n * new_median
        min_range = median - n * new_median
        dataframe[i]=pd.DataFrame(np.clip(dataframe[i].values, min_range, max_range),columns=None)

    return dataframe
def filter_extreme_3sigma(dataframe,n=3):
    # 3sigma ???
    for i in dataframe.columns:
        mean=dataframe[i].mean()
        std=dataframe[i].std()
        max_range=mean+n*std
        min_range=mean-n*std
        dataframe[i] = pd.DataFrame(np.clip(dataframe[i].values, min_range, max_range), columns=None)
    return dataframe

def filter_extreme_percentile(dataframe,min=0.025,max=0.975):
    # ???? ???
    for i in dataframe.columns:
        Temp=dataframe[i].sort_values()
        q=Temp.quantile([min,max])
        dataframe[i] = pd.DataFrame(np.clip(dataframe[i].values, q.iloc[0], q.iloc[1]), columns=None)
    return

In [ ]:
df=filter_extreme_3sigma(df)

In [ ]:
df.isnull().sum()

In [ ]:
import sys
!{sys.executable} -m pip install xgboost -i https://pypi.tuna.tsinghua.edu.cn/simple
!{sys.executable} -m pip install lightgbm -i https://pypi.tuna.tsinghua.edu.cn/simple
!{sys.executable} -m pip install matplotlib -i https://pypi.tuna.tsinghua.edu.cn/simple

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,f1_score
from sklearn.model_selection import train_test_split

In [ ]:

# X=df[['SOC?????', 'SIC?????',
#        'STC????', '??N', '??C/N?',  '???',
#        '?/??', '??(g)', '??(g)', 'plot__G11', 'plot__G12', 'plot__G13',
#        'plot__G16', 'plot__G17', 'plot__G18', 'plot__G19', 'plot__G20',
#        'plot__G21', 'plot__G6', 'plot__G8', 'plot__G9', '??__??', '??__?????',
#        '??__?????', '??__?????', '??__?????', '??__??(6?)', '??__??(0?)',
#        '??__??(3?)', '??__??(12?)', '???__AB', '???__PB', '???__PF',
#        '???__PR']]




X=df[['SOC?????', 'SIC?????',
       'STC????', '??N', '??C/N?']]

y=df['????']
# ?????????
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=12)
    
#?? ??
model = LogisticRegression()

model.fit(x_train, y_train)
print('?? ??')
print(classification_report(model.predict(x_test),y_test))


# Randomforest??

model = RandomForestClassifier()

model.fit(x_train, y_train)
print('Randomforest??')
print(classification_report(model.predict(x_test),y_test))

# lgbm??
model = lgb.LGBMClassifier()
model.fit(x_train, y_train)
print('lgbm??')
print(classification_report(model.predict(x_test),y_test))


# XGboost??
model = xgb.XGBClassifier()
model.fit(x_train, y_train)
print('XGboost??')
print(classification_report(model.predict(x_test),y_test))

In [ ]:
#?? ??
model = LogisticRegression()

model.fit(X, y)
print('?? ??')
print(classification_report(model.predict(X),y))

In [ ]:
model.coef_

In [ ]:
model.intercept_